In [1]:
import numpy as np 
import pylab as plt 
import numpy as np 
from scipy import linalg
import math 
# save_imgs_as_arrays(img1)
# save_imgs_as_arrays_1(img2)
corr_arrays = []

In [2]:
def normxcorr2(A,B, norm=True, mode='same'):

    from scipy.signal import correlate2d
    from skimage.feature import match_template
        
    A_ = A-np.nanmean(A)
    B_ = B-np.nanmean(B)
        
    return correlate2d(A_,B_, mode=mode)

In [3]:
def nd_xcorr_lag(img1, img2, lag=1, mode='same', demean=False):
        
    from scipy.signal import correlate
    import numpy as np 

    img1_ = img1[...,lag:img1.shape[-1]].copy()
    img2_ = img2[...,:img2.shape[-1]-lag].copy()

#     corr_a_b = correlate(img1_-img1_.mean(axis=-1)[...,None], 
#                          img2_-img2_.mean(axis=-1)[...,None], mode=mode)
    # corr_a_b = correlate(img1_, img2_, mode=mode)
    if demean:
        corr_a_b = correlate(img2_-img2_.mean(axis=-1)[...,None], 
                             img1_-img1_.mean(axis=-1)[...,None], 
                             mode=mode)
    else:
        corr_a_b = correlate(img2_, 
                             img1_, mode=mode)
    # corr_a_b = corr_a_b.mean(axis=-1) # integrate over time!. 
    corr_a_b = np.nanmax(corr_a_b, axis=-1)
    
    return corr_a_b

In [4]:
class PCCA_GC_Calculator:
    
    import math
    import numpy as np
    from scipy import linalg

    def __init__(self, X, Y_cause):
        """data for test whether the time series Y_cause causes X
        :param X: dim(X) = (N, dm) = (length of time series, dimension of X)
        :param Y_cause:
        """
        self.X = X
        self.Y = Y_cause
        self.X_t = X.T
        self.Y_t = Y_cause.T

    def calcSigmaHat(self, sigma, eta):
        return sigma + eta * np.identity(sigma.shape[0])

    def calcGrangerCausality(self, k, m, eta_xt=0.00001, eta_yt= 0.00001, eta_xtkm=0.00001):
        """
        :param k:
        :param m:
        :param eta_xt:
        :param eta_yt:
        :param eta_xtkm:
        :return:
        """
        N = self.X.shape[0]
        dim_X = self.X.shape[1]
        dim_Y = self.Y.shape[1]


        x_t = []
        y_t = []
        x_tk_m = []
        y_tk_m = []

        for t in range(k + m - 1, N):
            x_t.append(self.X[t])
            y_t.append(self.Y[t])

            cut_x = self.X[t - k - m + 1: t - k + 1]
            x_tk_m.append(np.ravel(cut_x[::-1]))         # reverse the array and make the array 1d array
            cut_y = self.Y[t - k - m + 1: t - k + 1]
            y_tk_m.append(np.ravel(cut_y[::-1]))         # reverse the array and make the array 1d array

        x_t = (np.array(x_t)).T
        y_t = (np.array(y_t)).T
        x_tk_m = (np.array(x_tk_m)).T
        y_tk_m = (np.array(y_tk_m)).T

        dim_x_t = x_t.shape[0]
        dim_y_t = y_t.shape[0]
        dim_x_tk_m = x_tk_m.shape[0]

        x = np.r_[x_t, y_t]
        y = np.r_[x_tk_m, y_tk_m]

        sigma = np.cov(m=x, y=y, rowvar=True)   # row of x and y represents a variable, and each column a single observation
        """ 
        sigma = ( sigma_xt_xt   sigma_xt_yt     sigma_xt_xtkm       sigma_xt_ytkm ) 
                ( sigma_yt_xt   sigma_yt_yt     sigma_yt_xtkm       sigma_yt_ytkm )
                ( sigma_xtkm_xt sigma_xtkm_yt   sigma_xtkm_xtkm     sigma_xtkm_ytkm )
                ( sigma_ytkm_xt sigma_ytkm_yt   sigma_ytkm_xtkm     sigma_ytkm_ytkm )
        """

        yt_start_idx = dim_x_t
        xtkm_start_idx = dim_x_t + dim_y_t
        ytkm_start_idx = dim_x_t + dim_y_t + dim_x_tk_m

        sigma_xt_xt = sigma[    0 : yt_start_idx, 0              : yt_start_idx]
        sigma_xt_xtkm = sigma[  0 : yt_start_idx, xtkm_start_idx : ytkm_start_idx]
        sigma_xt_ytkm = sigma[  0 : yt_start_idx, ytkm_start_idx :]

        sigma_xtkm_xt = sigma[  xtkm_start_idx : ytkm_start_idx, 0 : yt_start_idx]
        sigma_xtkm_xtkm = sigma[xtkm_start_idx : ytkm_start_idx, xtkm_start_idx : ytkm_start_idx]
        sigma_xtkm_ytkm = sigma[xtkm_start_idx : ytkm_start_idx, ytkm_start_idx : ]

        sigma_ytkm_xt = sigma[  ytkm_start_idx:, 0              : yt_start_idx]
        sigma_ytkm_xtkm = sigma[ytkm_start_idx:, xtkm_start_idx : ytkm_start_idx]
        sigma_ytkm_ytkm = sigma[ytkm_start_idx:, ytkm_start_idx : ]


        sigma_tilde_ytkm_xt_xtkm = sigma_ytkm_xt\
                                 - np.dot(np.dot(2 * sigma_ytkm_xtkm, np.linalg.inv(self.calcSigmaHat(sigma=sigma_xtkm_xtkm, eta=eta_xtkm))), sigma_xtkm_xt)\
                                 + np.dot(np.dot(np.dot(np.dot(sigma_ytkm_xtkm, np.linalg.inv(self.calcSigmaHat(sigma=sigma_xtkm_xtkm, eta=eta_xtkm))),sigma_xtkm_xtkm), np.linalg.inv(self.calcSigmaHat(sigma=sigma_xtkm_xtkm, eta=eta_xtkm))), sigma_xtkm_xt)

        sigma_tilde_xt_xt_xtkm = sigma_xt_xt \
                                   - np.dot(np.dot(2 * sigma_xt_xtkm, np.linalg.inv(self.calcSigmaHat(sigma=sigma_xtkm_xtkm, eta=eta_xtkm))), sigma_xtkm_xt) \
                                   + np.dot(np.dot(np.dot(np.dot(sigma_xt_xtkm, np.linalg.inv(self.calcSigmaHat(sigma=sigma_xtkm_xtkm, eta=eta_xtkm))),sigma_xtkm_xtkm), np.linalg.inv(self.calcSigmaHat(sigma=sigma_xtkm_xtkm, eta=eta_xtkm))), sigma_xtkm_xt)

        sigma_tilde_xt_ytkm_xtkm = sigma_xt_ytkm \
                                   - np.dot(np.dot(2 * sigma_xt_xtkm, np.linalg.inv(self.calcSigmaHat(sigma=sigma_xtkm_xtkm, eta=eta_xtkm))), sigma_xtkm_ytkm) \
                                   + np.dot(np.dot(np.dot(np.dot(sigma_xt_xtkm, np.linalg.inv(self.calcSigmaHat(sigma=sigma_xtkm_xtkm, eta=eta_xtkm))),sigma_xtkm_xtkm), np.linalg.inv(self.calcSigmaHat(sigma=sigma_xtkm_xtkm, eta=eta_xtkm))), sigma_xtkm_ytkm)

        sigma_tilde_ytkm_ytkm_xtkm = sigma_ytkm_ytkm \
                                   - np.dot(np.dot(2 * sigma_ytkm_xtkm, np.linalg.inv(self.calcSigmaHat(sigma=sigma_xtkm_xtkm, eta=eta_xtkm))), sigma_xtkm_ytkm) \
                                   + np.dot(np.dot(np.dot(np.dot(sigma_ytkm_xtkm, np.linalg.inv(self.calcSigmaHat(sigma=sigma_xtkm_xtkm, eta=eta_xtkm))),sigma_xtkm_xtkm), np.linalg.inv(self.calcSigmaHat(sigma=sigma_xtkm_xtkm, eta=eta_xtkm))), sigma_xtkm_ytkm)

        A = np.dot(np.dot(sigma_tilde_ytkm_xt_xtkm, np.linalg.inv(sigma_tilde_xt_xt_xtkm + eta_xt * np.identity(sigma_tilde_xt_xt_xtkm.shape[0]))), sigma_tilde_xt_ytkm_xtkm)
        B = sigma_tilde_ytkm_ytkm_xtkm + eta_yt * np.identity(sigma_tilde_ytkm_ytkm_xtkm.shape[0])

        eigenvalues = np.real(linalg.eig(a=A, b=B)[0])
        eigenvalue = np.max(eigenvalues)
        if eigenvalue > 1.0:
            eigenvalue = 0.9999
        Gyx = 0.5 * math.log(1 / (1 - eigenvalue), 2)

        return Gyx


In [5]:
def pcca_cause(img1, img2, 
                k=1, m=3, 
                eta_xt=5e-4, 
                eta_yt=5e-4,
                eta_xtkm=5e-4):
    # img1 = np.load(f'{number}.npy')
    # img2 = np.load(f'{number}_1.npy')
    Y = img1.copy() #- np.mean(img1)
    X = img2.copy() #- np.mean(img1)

    Y = Y.reshape(-1, Y.shape[-1]).T
    X = X.reshape(-1, X.shape[-1]).T
    calc_xy = PCCA_GC_Calculator(X=X, Y_cause=Y)
    # Gy_to_x = calc_xy.calcGrangerCausality(k=1, m=1,
    #                                        eta_xt=1e-5, eta_yt=1e-5, eta_xtkm=1e-5) # delay lag=1 and order=1 # this is slow.... 
    Gy_to_x = calc_xy.calcGrangerCausality(k=k, m=m,
                                           eta_xt=eta_xt, 
                                           eta_yt=eta_yt, 
                                           eta_xtkm=eta_xtkm) # etas are very important 
    return Gy_to_x


In [ ]:
import numpy as np
import os

# Directory where the .npy files are located
directory_path = "../DiME"

# Get a list of all .npy files in the directory that end with "ce.npy"
file_list = [f for f in os.listdir(directory_path) if f.endswith("x_t.npy")]

# Initialize an empty list to store the loaded arrays
loaded_arrays = []

# Loop through the file list, load each array, and append it to loaded_arrays
for file_name in file_list:
    file_path = os.path.join(directory_path, file_name)
    loaded_array = np.load(file_path)
    
    loaded_arrays.append(loaded_array)

# Concatenate the loaded arrays along a specified axis (e.g., axis=0 for vertical concatenation)
concatenated_array = np.concatenate(loaded_arrays, axis=0)
print(concatenated_array.shape)
# Now 'concatenated_array' contains the concatenated data from all the .npy files


In [6]:
img1 = np.load(f'test.npy')
img2 = np.load(f'test1.npy')
# plt.imshow(img1)
# plt.axis('off')  # Optional: Turn off axis labels and ticks
# plt.show()
# img1 = np.nan_to_num(img1)
# img2 = np.nan_to_num(img2)
ce0 = pcca_cause(img1, img2)

In [7]:
import numpy as np
import torch
from PIL import Image
from IPython.display import display

# denorm_fn = lambda x: x * 0.5 + 0.5
# img = denorm_fn(ce0)
# # img = np.transpose(img, axes=(0, 2, 3, 1))
# img = (img * 255).astype('uint8')
# image = Image.fromarray(img)
# image.show()
print(ce0)
# plt.imshow(ce0, cmap='gray')  # For grayscale images
# plt.show()
# # Save and display each image
# for idx, i in enumerate(img):
#     i = Image.fromarray(i)
#     # i.save(f'{idx}.png')
#     display(i)

0.43827101972883853


In [8]:
def differential_covariance(X, eps=1e-12, alpha=1e-3):
    """ uses ridge regularization 
    """
    import numpy as np 
    
    # standardize
    # X_ = (X - np.nanmean(X, axis=1)[:,None])  / ( np.nanstd(X, axis=1)[:,None] + eps )
    X_ = X.copy()
    
    # # differential 
    # X_pad = np.pad(X_, pad_width=[[0,0],[1,1]], mode='edge')
    # dX_ = (X_pad[:,2:] - X_pad[:,:-2]) / 2.
    # # dX_ = np.gradient(X_, axis=1)
    dX_ = X_[:,1:] - X_[:,:-1]
    X_ = X_[:,1:].copy()
    # print(X_.shape,dX_.shape)
    X_ = X_.T
    dX_ = dX_.T
    
    # linear least squares solution . 
    dX_X = dX_.T.dot(X_)
    X_X = X_.T.dot(X_)
    
    # W = np.linalg.solve(X_X+reg*np.eye(len(X_X)), dX_X) # transpose... 
    W = dX_X.dot(np.linalg.inv(X_X +alpha*np.eye(len(X_X))))
    
    return W 

In [9]:
for number in range(0, 3):
    img1 = np.load(f'{number}.npy')
    img2 = np.load(f'{number}_1.npy')
    print('mask1 done')
    # compile all the timeseries
    Y_ = np.array([img1, 
                   img2])
    # Y_ = Y_.reshape(len(Y_), -1, Y_.shape[-1])
    print(Y_.shape)
    """
    Compute the diff covariance 
    """
    print('Here')
    print(Y_.reshape(-1, Y_.shape[-1]).shape)
    W_ = differential_covariance(Y_.reshape(-1, Y_.shape[-1]), eps=1e-12, alpha=1e-3)
    print('W_')
    # W_ = W_.T.copy()
    N = Y_.shape[1] # this is the flattened over spatial windows. 
    N_rows = int(np.sqrt(N))

    # W_out = W_[1:,0].copy() # - W_[0,1:]
    W_out = W_[:N, N:].copy()
    corr_array = np.reshape(W_out,axes=(3,3,128,128))
#         corr_array = np.zeros((N_rows,N_rows))

#         for ii in np.arange(N_rows):
#             for jj in np.arange(N_rows):
#                 ind = ii*N_rows + jj
#                 corr_array[ii,jj] = W_out[ind,ind]
    corr_arrays.append(corr_array)
    print(corr_arrays.shape)

return corr_array, dil_mask

FileNotFoundError: [Errno 2] No such file or directory: '0.npy'

In [ ]:
corr_arrays = np.transpose(corr_arrays, axes=(0, 3, 1, 2)).sum(dim=1, keepdim=True)
corr_array = corr_array.reshape((winsize,winsize))
mid = corr_array.shape[1]//2

corr_x_direction = -np.nansum(corr_array[:,:mid]) + np.nansum(corr_array[:,mid+1:])
corr_y_direction = -np.nansum(corr_array[:mid]) + np.nansum(corr_array[mid+1:])
intensity = np.nansum(corr_array) #* np.sqrt(corr_x_direction**2 + corr_y_direction**2)

mean_vector = np.hstack([corr_y_direction, corr_x_direction])
mean_vector = mean_vector * intensity

# mask = torch.from_numpy(corr_arrays)
# dil_mask = F.max_pool2d(mask,
#                     dilation, stride=1,
#                     padding=(dilation - 1) // 2)
